In [1]:
# ===========================
# Step 1: Import Libraries
# ===========================

import pandas as pd
import nltk
import re

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ===========================
# Step 2: Download Stopwords
# ===========================

nltk.download('stopwords')


# ===========================
# Step 3: Load Dataset
# ===========================

data = pd.read_csv("IMDB Dataset.csv")

print("First 5 Rows:")
print(data.head())

print("\nDataset Shape:")
print(data.shape)


# ===========================
# Step 4: Initialize
# ===========================

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()


# ===========================
# Step 5: Text Preprocessing
# ===========================

def preprocess(text):

    # Convert to lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenization
    words = text.split()

    # Remove stopwords and apply stemming
    words = [
        stemmer.stem(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)


# ===========================
# Step 6: Apply Preprocessing
# ===========================

data["Clean_Text"] = data["review"].apply(preprocess)

print("\nCleaned Reviews:")
print(data[["review", "Clean_Text"]].head())


# ===========================
# Step 7: TF-IDF Vectorization
# ===========================

tfidf = TfidfVectorizer()

X = tfidf.fit_transform(data["Clean_Text"])

y = data["sentiment"]

print("\nTF-IDF Matrix Shape:")
print(X.shape)


# ===========================
# Step 8: Split Dataset
# ===========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])


# ===========================
# Step 9: Train Model
# ===========================

model = MultinomialNB()

model.fit(X_train, y_train)

print("\nModel Training Completed!")


# ===========================
# Step 10: Prediction
# ===========================

prediction = model.predict(X_test)

print("\nPredicted Values:")
print(prediction[:20])


# ===========================
# Step 11: Evaluation
# ===========================

accuracy = accuracy_score(y_test, prediction)

precision = precision_score(
    y_test,
    prediction,
    pos_label='positive'
)

recall = recall_score(
    y_test,
    prediction,
    pos_label='positive'
)

f1 = f1_score(
    y_test,
    prediction,
    pos_label='positive'
)

print("\nAccuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

print("\nClassification Report:")
print(classification_report(y_test, prediction))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, prediction))


# ===========================
# Step 12: Predict New Review
# ===========================

new_review = [
    "This movie was absolutely fantastic. I loved every scene."
]

clean_review = [
    preprocess(review)
    for review in new_review
]

new_text = tfidf.transform(clean_review)

result = model.predict(new_text)

print("\nReview:", new_review[0])
print("Predicted Sentiment:", result[0])


# ===========================
# Step 13: Another Example
# ===========================

new_review2 = [
    "This was the worst movie I have ever watched."
]

clean_review2 = [
    preprocess(review)
    for review in new_review2
]

new_text2 = tfidf.transform(clean_review2)

result2 = model.predict(new_text2)

print("\nReview:", new_review2[0])
print("Predicted Sentiment:", result2[0])

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Elaki\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


First 5 Rows:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Dataset Shape:
(50000, 2)

Cleaned Reviews:
                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                          Clean_Text  
0  one review mention watch oz episod hook right ...  
1  wonder littl product film techniqu unassum old...  
2  thought wonder way spend time hot summer weeke...  
3  bas